In [1]:
# --- repo bootstrap: make src/ + this domain's config importable, run from repo root ---
#   src/                    shared library, in every container
#   domains/credit_risk/    THIS domain's model_config.py — only in the
#                           credit-risk containers, so the domain-agnostic stages
#                           physically cannot import domain settings
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
sys.path.insert(0, str(_ROOT / 'domains' / 'credit_risk'))
os.chdir(_ROOT)

In [2]:
# ⚠️ TEMPORARY — MLflow logging called from the notebook.
# To be removed once mtr.run_classifier / run_regressor / run_cox / etc. call
# mt.log_model_result() internally. Until then every model needs an explicit
# log call in its own cell (see below).
import json
import mlflow_tracking as mt

SPLIT_META = json.load(open("transformed_data/model_split_meta.json"))
RUN_TAGS = {"source": "live"}          # distinguishes these from the backfill

# Force a fresh connection probe and print the verdict. _connect() caches its
# ON/OFF answer module-wide, so a kernel that first imported this before the
# server was up keeps no-opping every log call below with no visible sign.
mt.init()

 MLflow ON  -> http://localhost:5001  (experiment 'credit_risk', id 1)


True

# Aave V3.1 — model training (baselines + tuned zoo + PRI meta-learner)

One cell per model: tune (embargoed walk-forward CV on train only), fit, persist,
and display the model's own scorecard (train / val / test — the train row is the
overfit gap; test carries a block-bootstrap AUC CI). F1 thresholds are tuned on
val and reused on test. Two baselines per task set the bar every model must beat.

Classifiers predict next 1/3/7-day liquidation stress (per-horizon train
thresholds); regressors predict log1p(next-day liquidation debt covered USD)
with clipped USD back-transform; IsolationForest, Cox PH, LSTM and a Temporal
Transformer add anomaly / hazard / sequence perspectives. The final cell blends
components (skill-shrunk weights, baselines excluded) into the PRI (0-100).
Cross-model staging lives in `model_results.ipynb`.

In [3]:
import warnings
warnings.filterwarnings("ignore")

import sklearn, xgboost, lightgbm
from IPython.display import display

import model_config as cfg
import model_training as mtr
import model_persistence as mps
import model_benchmarks as mbench

print(f" sklearn {sklearn.__version__} | xgboost {xgboost.__version__} | "
      f"lightgbm {lightgbm.__version__}")
print(f" lifelines available: {cfg.LIFELINES_AVAILABLE} | torch available: {cfg.TORCH_AVAILABLE}")

 sklearn 1.9.0 | xgboost 3.3.0 | lightgbm 4.6.0
 lifelines available: True | torch available: True


In [4]:
data = mps.load_model_data()
manifest = mps.write_run_manifest(data)
for t in cfg.STRESS_TARGETS:
    rates = {s: round(float(data["y"][t][s].mean()), 3) for s in ("train", "val", "test")}
    print(f" {t}: positive rate {rates}")

 loaded splits [189, 82, 72] rows, 53 features (21 log1p, 0 dropped)
 wrote run_manifest.json → model_results/
 y_stress_1d: positive rate {'train': 0.249, 'val': 0.341, 'test': 0.403}
 y_stress_3d: positive rate {'train': 0.249, 'val': 0.366, 'test': 0.375}
 y_stress_7d: positive rate {'train': 0.243, 'val': 0.378, 'test': 0.736}


## Baselines — the bar every model must beat

In [5]:
for res in mtr.run_baseline_classifiers(data):
    mps.save_model_result(res)
    mt.log_model_result(res, SPLIT_META, RUN_TAGS)
    display(mbench.model_scorecard(res))

 baseline_persistence: test ROC-AUC 0.509 | PR-AUC 0.407
 baseline_prevalence: test ROC-AUC 0.500 | PR-AUC 0.403
 saved baseline_persistence__classifier → model_results/
    mlflow: logged baseline_persistence__classifier (47 metrics) run cadfbf6e
🏃 View run baseline_persistence__classifier at: http://localhost:5001/#/experiments/1/runs/cadfbf6e19c94437925cafdc583b4669
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.579,0.292,0.398,None,47,189,None,None
1,y_stress_1d,val,0.611,0.412,0.509,None,28,82,None,None
2,y_stress_1d,test,0.509,0.407,0.574,None,29,72,None,"[0.379, 0.681]"
3,y_stress_3d,train,0.550,0.273,0.398,None,47,189,None,None
4,y_stress_3d,val,0.589,0.420,0.536,None,30,82,None,None
5,y_stress_3d,test,0.533,0.392,0.545,None,27,72,None,None
6,y_stress_7d,train,0.555,0.270,0.391,None,46,189,None,None
7,y_stress_7d,val,0.631,0.468,0.549,None,31,82,None,None
8,y_stress_7d,test,0.559,0.761,0.848,None,53,72,None,None


 saved baseline_prevalence__classifier → model_results/
    mlflow: logged baseline_prevalence__classifier (47 metrics) run c7728fc6
🏃 View run baseline_prevalence__classifier at: http://localhost:5001/#/experiments/1/runs/c7728fc6a7fb49328ae2f2169dc25647
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.5,0.249,0.398,None,47,189,None,None
1,y_stress_1d,val,0.5,0.341,0.509,None,28,82,None,None
2,y_stress_1d,test,0.5,0.403,0.574,None,29,72,None,"[0.500, 0.500]"
3,y_stress_3d,train,0.5,0.249,0.398,None,47,189,None,None
4,y_stress_3d,val,0.5,0.366,0.536,None,30,82,None,None
5,y_stress_3d,test,0.5,0.375,0.545,None,27,72,None,None
6,y_stress_7d,train,0.5,0.243,0.391,None,46,189,None,None
7,y_stress_7d,val,0.5,0.378,0.549,None,31,82,None,None
8,y_stress_7d,test,0.5,0.736,0.848,None,53,72,None,None


In [6]:
for res in mtr.run_baseline_regressors(data):
    mps.save_model_result(res)
    mt.log_model_result(res, SPLIT_META, RUN_TAGS)
    display(mbench.model_scorecard(res))

 baseline_persistence: test RMSE(log) 5.965 | R² -0.737 | MAE $6,856,707
 baseline_median: test RMSE(log) 4.683 | R² -0.070 | MAE $6,161,416
 saved baseline_persistence__regressor → model_results/
    mlflow: logged baseline_persistence__regressor (12 metrics) run 8ee0cfa2
🏃 View run baseline_persistence__regressor at: http://localhost:5001/#/experiments/1/runs/8ee0cfa2222b4116a5e011e9b5d51a7f
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,5.317,4.238,-0.373,1087083.0
1,val,6.451,5.211,-0.485,4200620.0
2,test,5.965,4.616,-0.737,6856707.0


 saved baseline_median__regressor → model_results/
    mlflow: logged baseline_median__regressor (12 metrics) run cdc025d1
🏃 View run baseline_median__regressor at: http://localhost:5001/#/experiments/1/runs/cdc025d188e84803b6e0d4508061595c
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,4.602,3.746,-0.028,834075.0
1,val,5.300,4.359,-0.003,3140725.0
2,test,4.683,3.825,-0.070,6161416.0


## Classifiers — next-period liquidation stress (1d / 3d / 7d), tuned

In [7]:
res_logistic_regression = mtr.run_classifier("logistic_regression", data)
mps.save_model_result(res_logistic_regression)
mt.log_model_result(res_logistic_regression, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_logistic_regression))

 logistic_regression: tuned over 3 configs -> {'C': 10.0} (CV AUC 0.499)
 logistic_regression: train/val/test ROC-AUC 0.852/0.478/0.431 [0.258,0.679] | CV 0.499±0.049
 saved logistic_regression__classifier → model_results/
    mlflow: logged logistic_regression__classifier (58 metrics) run 25328e4e
🏃 View run logistic_regression__classifier at: http://localhost:5001/#/experiments/1/runs/25328e4eaa024ae2841b87fe4e3478ae
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.852,0.577,0.418,0.163,47,189,0.499,None
1,y_stress_1d,val,0.478,0.385,0.519,0.394,28,82,0.499,None
2,y_stress_1d,test,0.431,0.476,0.521,0.507,29,72,0.499,"[0.258, 0.679]"
3,y_stress_3d,train,0.896,0.723,0.720,0.129,47,189,NaN,None
4,y_stress_3d,val,0.607,0.431,0.578,0.392,30,82,NaN,None
5,y_stress_3d,test,0.704,0.626,0.581,0.476,27,72,NaN,None
6,y_stress_7d,train,0.918,0.791,0.646,0.115,46,189,NaN,None
7,y_stress_7d,val,0.537,0.427,0.569,0.453,31,82,NaN,None
8,y_stress_7d,test,0.669,0.845,0.822,0.254,53,72,NaN,None


In [8]:
res_ridge_classifier = mtr.run_classifier("ridge_classifier", data)
mps.save_model_result(res_ridge_classifier)
mt.log_model_result(res_ridge_classifier, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_ridge_classifier))

 ridge_classifier: tuned over 3 configs -> {'alpha': 0.1} (CV AUC 0.561)
 ridge_classifier: train/val/test ROC-AUC 0.869/0.411/0.465 [0.328,0.677] | CV 0.561±0.094
 saved ridge_classifier__classifier → model_results/
    mlflow: logged ridge_classifier__classifier (49 metrics) run 710cf0a2
🏃 View run ridge_classifier__classifier at: http://localhost:5001/#/experiments/1/runs/710cf0a26bad46d189e07387d6178c14
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.869,0.600,0.398,None,47,189,0.561,None
1,y_stress_1d,val,0.411,0.299,0.509,None,28,82,0.561,None
2,y_stress_1d,test,0.465,0.466,0.574,None,29,72,0.561,"[0.328, 0.677]"
3,y_stress_3d,train,0.907,0.744,0.604,None,47,189,NaN,None
4,y_stress_3d,val,0.522,0.361,0.542,None,30,82,NaN,None
5,y_stress_3d,test,0.669,0.504,0.559,None,27,72,NaN,None
6,y_stress_7d,train,0.921,0.782,0.656,None,46,189,NaN,None
7,y_stress_7d,val,0.517,0.418,0.561,None,31,82,NaN,None
8,y_stress_7d,test,0.679,0.842,0.777,None,53,72,NaN,None


In [9]:
res_gaussian_nb = mtr.run_classifier("gaussian_nb", data)
mps.save_model_result(res_gaussian_nb)
mt.log_model_result(res_gaussian_nb, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_gaussian_nb))

 gaussian_nb: tuned over 3 configs -> {'var_smoothing': 1e-09} (CV AUC 0.484)
 gaussian_nb: train/val/test ROC-AUC 0.617/0.604/0.461 [0.379,0.549] | CV 0.484±0.102
 saved gaussian_nb__classifier → model_results/
    mlflow: logged gaussian_nb__classifier (58 metrics) run ab806a6f
🏃 View run gaussian_nb__classifier at: http://localhost:5001/#/experiments/1/runs/ab806a6f3dd44fc8af9dcb6e83192dc6
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.617,0.366,0.396,0.196,47,189,0.484,None
1,y_stress_1d,val,0.604,0.450,0.583,0.248,28,82,0.484,None
2,y_stress_1d,test,0.461,0.383,0.314,0.313,29,72,0.484,"[0.379, 0.549]"
3,y_stress_3d,train,0.624,0.370,0.400,0.211,47,189,NaN,None
4,y_stress_3d,val,0.510,0.407,0.541,0.285,30,82,NaN,None
5,y_stress_3d,test,0.570,0.485,0.536,0.300,27,72,NaN,None
6,y_stress_7d,train,0.603,0.376,0.417,0.204,46,189,NaN,None
7,y_stress_7d,val,0.533,0.398,0.559,0.311,31,82,NaN,None
8,y_stress_7d,test,0.401,0.667,0.829,0.614,53,72,NaN,None


In [10]:
res_kneighbors = mtr.run_classifier("kneighbors", data)
mps.save_model_result(res_kneighbors)
mt.log_model_result(res_kneighbors, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_kneighbors))

 kneighbors: tuned over 3 configs -> {'n_neighbors': 15} (CV AUC 0.524)
 kneighbors: train/val/test ROC-AUC 0.691/0.583/0.521 [0.352,0.660] | CV 0.524±0.094
 saved kneighbors__classifier → model_results/
    mlflow: logged kneighbors__classifier (58 metrics) run a2c4c36d
🏃 View run kneighbors__classifier at: http://localhost:5001/#/experiments/1/runs/a2c4c36d592142748d6c39d270438b1e
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.691,0.359,0.489,0.172,47,189,0.524,None
1,y_stress_1d,val,0.583,0.438,0.523,0.234,28,82,0.524,None
2,y_stress_1d,test,0.521,0.434,0.481,0.276,29,72,0.524,"[0.352, 0.660]"
3,y_stress_3d,train,0.754,0.459,0.537,0.161,47,189,NaN,None
4,y_stress_3d,val,0.579,0.434,0.551,0.248,30,82,NaN,None
5,y_stress_3d,test,0.563,0.491,0.456,0.256,27,72,NaN,None
6,y_stress_7d,train,0.795,0.521,0.391,0.154,46,189,NaN,None
7,y_stress_7d,val,0.532,0.396,0.549,0.267,31,82,NaN,None
8,y_stress_7d,test,0.517,0.761,0.848,0.488,53,72,NaN,None


In [11]:
res_svc_rbf = mtr.run_classifier("svc_rbf", data)
mps.save_model_result(res_svc_rbf)
mt.log_model_result(res_svc_rbf, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_svc_rbf))

 svc_rbf: tuned over 3 configs -> {'C': 2.0} (CV AUC 0.577)
 svc_rbf: train/val/test ROC-AUC 0.976/0.490/0.501 [0.365,0.660] | CV 0.577±0.117
 saved svc_rbf__classifier → model_results/
    mlflow: logged svc_rbf__classifier (58 metrics) run e7dfec85
🏃 View run svc_rbf__classifier at: http://localhost:5001/#/experiments/1/runs/e7dfec8592f94db6b1b55b2aefef8fc6
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.976,0.891,0.437,0.165,47,189,0.577,None
1,y_stress_1d,val,0.490,0.371,0.514,0.233,28,82,0.577,None
2,y_stress_1d,test,0.501,0.383,0.611,0.265,29,72,0.577,"[0.365, 0.660]"
3,y_stress_3d,train,0.968,0.894,0.759,0.110,47,189,NaN,None
4,y_stress_3d,val,0.573,0.451,0.552,0.235,30,82,NaN,None
5,y_stress_3d,test,0.512,0.401,0.457,0.249,27,72,NaN,None
6,y_stress_7d,train,0.974,0.911,0.752,0.071,46,189,NaN,None
7,y_stress_7d,val,0.481,0.430,0.566,0.260,31,82,NaN,None
8,y_stress_7d,test,0.476,0.729,0.796,0.377,53,72,NaN,None


In [12]:
res_random_forest = mtr.run_classifier("random_forest", data)
mps.save_model_result(res_random_forest)
mt.log_model_result(res_random_forest, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_random_forest))

 random_forest: tuned over 4 configs -> {'max_depth': 6, 'min_samples_leaf': 5} (CV AUC 0.597)
 random_forest: train/val/test ROC-AUC 0.995/0.616/0.468 [0.338,0.614] | CV 0.597±0.049
 saved random_forest__classifier → model_results/
    mlflow: logged random_forest__classifier (58 metrics) run 86e7ab8a
🏃 View run random_forest__classifier at: http://localhost:5001/#/experiments/1/runs/86e7ab8a4a31497cbef2f93b48d0c7ae
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.995,0.986,0.718,0.106,47,189,0.597,None
1,y_stress_1d,val,0.616,0.486,0.545,0.231,28,82,0.597,None
2,y_stress_1d,test,0.468,0.400,0.486,0.259,29,72,0.597,"[0.338, 0.614]"
3,y_stress_3d,train,0.994,0.985,0.482,0.097,47,189,NaN,None
4,y_stress_3d,val,0.586,0.508,0.545,0.242,30,82,NaN,None
5,y_stress_3d,test,0.514,0.536,0.531,0.255,27,72,NaN,None
6,y_stress_7d,train,0.999,0.997,0.672,0.094,46,189,NaN,None
7,y_stress_7d,val,0.571,0.422,0.558,0.252,31,82,NaN,None
8,y_stress_7d,test,0.474,0.775,0.817,0.284,53,72,NaN,None


In [13]:
res_extra_trees = mtr.run_classifier("extra_trees", data)
mps.save_model_result(res_extra_trees)
mt.log_model_result(res_extra_trees, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_extra_trees))

 extra_trees: tuned over 4 configs -> {'max_depth': None, 'min_samples_leaf': 5} (CV AUC 0.620)
 extra_trees: train/val/test ROC-AUC 0.986/0.613/0.515 [0.364,0.670] | CV 0.620±0.069
 saved extra_trees__classifier → model_results/
    mlflow: logged extra_trees__classifier (58 metrics) run 2969d391
🏃 View run extra_trees__classifier at: http://localhost:5001/#/experiments/1/runs/2969d391ca604defad4547f76564c3e1
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.986,0.963,0.723,0.124,47,189,0.62,None
1,y_stress_1d,val,0.613,0.536,0.560,0.227,28,82,0.62,None
2,y_stress_1d,test,0.515,0.457,0.486,0.253,29,72,0.62,"[0.364, 0.670]"
3,y_stress_3d,train,0.986,0.963,0.485,0.113,47,189,NaN,None
4,y_stress_3d,val,0.601,0.509,0.556,0.239,30,82,NaN,None
5,y_stress_3d,test,0.509,0.558,0.516,0.262,27,72,NaN,None
6,y_stress_7d,train,0.996,0.987,0.860,0.113,46,189,NaN,None
7,y_stress_7d,val,0.636,0.486,0.606,0.232,31,82,NaN,None
8,y_stress_7d,test,0.464,0.773,0.400,0.301,53,72,NaN,None


In [14]:
res_hist_gradient_boosting = mtr.run_classifier("hist_gradient_boosting", data)
mps.save_model_result(res_hist_gradient_boosting)
mt.log_model_result(res_hist_gradient_boosting, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_hist_gradient_boosting))

 hist_gradient_boosting: tuned over 4 configs -> {'learning_rate': 0.05, 'max_leaf_nodes': 7} (CV AUC 0.541)
 hist_gradient_boosting: train/val/test ROC-AUC 1.000/0.620/0.548 [0.405,0.673] | CV 0.541±0.089
 saved hist_gradient_boosting__classifier → model_results/
    mlflow: logged hist_gradient_boosting__classifier (58 metrics) run 7151b602
🏃 View run hist_gradient_boosting__classifier at: http://localhost:5001/#/experiments/1/runs/7151b60270e847fa801e1d4abe7e9d7a
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,1.000,1.000,0.969,0.031,47,189,0.541,None
1,y_stress_1d,val,0.620,0.515,0.576,0.235,28,82,0.541,None
2,y_stress_1d,test,0.548,0.423,0.261,0.287,29,72,0.541,"[0.405, 0.673]"
3,y_stress_3d,train,1.000,1.000,0.989,0.030,47,189,NaN,None
4,y_stress_3d,val,0.591,0.463,0.554,0.253,30,82,NaN,None
5,y_stress_3d,test,0.552,0.498,0.359,0.253,27,72,NaN,None
6,y_stress_7d,train,1.000,1.000,0.594,0.022,46,189,NaN,None
7,y_stress_7d,val,0.459,0.367,0.549,0.329,31,82,NaN,None
8,y_stress_7d,test,0.487,0.769,0.829,0.324,53,72,NaN,None


In [15]:
res_xgboost = mtr.run_classifier("xgboost", data)
mps.save_model_result(res_xgboost)
mt.log_model_result(res_xgboost, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_xgboost))

 xgboost: tuned over 4 configs -> {'max_depth': 4, 'learning_rate': 0.03} (CV AUC 0.632)
 xgboost: train/val/test ROC-AUC 1.000/0.602/0.484 [0.364,0.602] | CV 0.632±0.076
 saved xgboost__classifier → model_results/
    mlflow: logged xgboost__classifier (58 metrics) run 430fcaf5
🏃 View run xgboost__classifier at: http://localhost:5001/#/experiments/1/runs/430fcaf587c5450999bc0480504cf4ab
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,1.000,1.000,0.461,0.002,47,189,0.632,None
1,y_stress_1d,val,0.602,0.535,0.523,0.240,28,82,0.632,None
2,y_stress_1d,test,0.484,0.377,0.580,0.349,29,72,0.632,"[0.364, 0.602]"
3,y_stress_3d,train,1.000,1.000,1.000,0.001,47,189,NaN,None
4,y_stress_3d,val,0.609,0.505,0.576,0.257,30,82,NaN,None
5,y_stress_3d,test,0.582,0.556,0.294,0.253,27,72,NaN,None
6,y_stress_7d,train,1.000,1.000,0.571,0.001,46,189,NaN,None
7,y_stress_7d,val,0.421,0.343,0.549,0.357,31,82,NaN,None
8,y_stress_7d,test,0.471,0.762,0.848,0.382,53,72,NaN,None


In [16]:
res_lightgbm = mtr.run_classifier("lightgbm", data)
mps.save_model_result(res_lightgbm)
mt.log_model_result(res_lightgbm, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_lightgbm))

 lightgbm: tuned over 4 configs -> {'max_depth': -1, 'learning_rate': 0.05} (CV AUC 0.599)
 lightgbm: train/val/test ROC-AUC 1.000/0.558/0.557 [0.399,0.699] | CV 0.599±0.082
 saved lightgbm__classifier → model_results/
    mlflow: logged lightgbm__classifier (58 metrics) run 10ccc505
🏃 View run lightgbm__classifier at: http://localhost:5001/#/experiments/1/runs/10ccc5050cf54edd91a5c786a6903d31
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,1.000,1.000,0.414,0.000,47,189,0.599,None
1,y_stress_1d,val,0.558,0.486,0.519,0.273,28,82,0.599,None
2,y_stress_1d,test,0.557,0.415,0.592,0.391,29,72,0.599,"[0.399, 0.699]"
3,y_stress_3d,train,1.000,1.000,0.505,0.000,47,189,NaN,None
4,y_stress_3d,val,0.577,0.467,0.541,0.294,30,82,NaN,None
5,y_stress_3d,test,0.513,0.421,0.521,0.334,27,72,NaN,None
6,y_stress_7d,train,1.000,1.000,0.601,0.000,46,189,NaN,None
7,y_stress_7d,val,0.478,0.398,0.549,0.410,31,82,NaN,None
8,y_stress_7d,test,0.428,0.707,0.848,0.439,53,72,NaN,None


In [17]:
res_mlp = mtr.run_classifier("mlp", data)
mps.save_model_result(res_mlp)
mt.log_model_result(res_mlp, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_mlp))

 mlp: tuned over 4 configs -> {'alpha': 0.001, 'hidden_layer_sizes': [64]} (CV AUC 0.575)
 mlp: train/val/test ROC-AUC 0.659/0.596/0.487 [0.368,0.632] | CV 0.575±0.086
 saved mlp__classifier → model_results/
    mlflow: logged mlp__classifier (58 metrics) run 94855bb5
🏃 View run mlp__classifier at: http://localhost:5001/#/experiments/1/runs/94855bb56d644ee3acea283c76c6e36e
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.659,0.366,0.434,0.206,47,189,0.575,None
1,y_stress_1d,val,0.596,0.496,0.537,0.242,28,82,0.575,None
2,y_stress_1d,test,0.487,0.474,0.427,0.252,29,72,0.575,"[0.368, 0.632]"
3,y_stress_3d,train,0.640,0.402,0.423,0.228,47,189,NaN,None
4,y_stress_3d,val,0.573,0.455,0.552,0.289,30,82,NaN,None
5,y_stress_3d,test,0.586,0.565,0.521,0.271,27,72,NaN,None
6,y_stress_7d,train,0.609,0.365,0.407,0.241,46,189,NaN,None
7,y_stress_7d,val,0.591,0.454,0.581,0.282,31,82,NaN,None
8,y_stress_7d,test,0.616,0.819,0.800,0.229,53,72,NaN,None


## Anomaly / survival / sequence models

In [18]:
res_iso = mtr.run_isolation_forest(data)
mps.save_model_result(res_iso)
mt.log_model_result(res_iso, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_iso))

 isolation_forest: test ROC-AUC 0.645 | PR-AUC 0.624 (anomaly score vs next-day stress — NOTE: features include same-day liquidation columns, so this partly measures target autocorrelation; judge it against baseline_persistence)
 saved isolation_forest__anomaly → model_results/
    mlflow: logged isolation_forest__anomaly (47 metrics) run 4b91dc08
🏃 View run isolation_forest__anomaly at: http://localhost:5001/#/experiments/1/runs/4b91dc0870604f0781a354b7ab37fd83
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.570,0.302,0.366,None,47,189,None,None
1,y_stress_1d,val,0.508,0.376,0.455,None,28,82,None,None
2,y_stress_1d,test,0.645,0.624,0.598,None,29,72,None,"[0.491, 0.803]"
3,y_stress_3d,train,0.591,0.369,0.366,None,47,189,None,None
4,y_stress_3d,val,0.558,0.428,0.511,None,30,82,None,None
5,y_stress_3d,test,0.647,0.570,0.547,None,27,72,None,None
6,y_stress_7d,train,0.618,0.353,0.383,None,46,189,None,None
7,y_stress_7d,val,0.493,0.405,0.484,None,31,82,None,None
8,y_stress_7d,test,0.570,0.788,0.826,None,53,72,None,None


In [19]:
res_cox = mtr.run_cox(data)
mps.save_model_result(res_cox)
mt.log_model_result(res_cox, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_cox))

 cox_ph: C-index train 0.674 | val 0.590 | test 0.501 (15 features)
 saved cox_ph__survival → model_results/
    mlflow: logged cox_ph__survival (3 metrics) run 214605af
🏃 View run cox_ph__survival at: http://localhost:5001/#/experiments/1/runs/214605afcbb24ff7b1497e37ca7739e3
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,c_index
0,train,0.674
1,val,0.590
2,test,0.501


In [20]:
res_lstm = mtr.run_lstm(data)
mps.save_model_result(res_lstm)
mt.log_model_result(res_lstm, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_lstm))

 lstm: train/val/test ROC-AUC 0.722/0.625/0.531 (lookback 14d)
 saved lstm__classifier → model_results/
    mlflow: logged lstm__classifier (20 metrics) run f77f7ad1
🏃 View run lstm__classifier at: http://localhost:5001/#/experiments/1/runs/f77f7ad18c0f43e9926af34d92ba36ba
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.722,0.427,0.474,0.241,42,176,None,None
1,y_stress_1d,val,0.625,0.455,0.575,0.239,28,82,None,None
2,y_stress_1d,test,0.531,0.483,0.513,0.246,29,72,None,"[0.311, 0.727]"


In [21]:
res_transformer = mtr.run_transformer(data)
mps.save_model_result(res_transformer)
mt.log_model_result(res_transformer, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_transformer))

 transformer: train/val/test ROC-AUC 0.710/0.639/0.554 (lookback 14d)
 saved transformer__classifier → model_results/
    mlflow: logged transformer__classifier (20 metrics) run 984466bb
🏃 View run transformer__classifier at: http://localhost:5001/#/experiments/1/runs/984466bbad3e4301983b7747ba687379
🧪 View experiment at: http://localhost:5001/#/experiments/1


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.710,0.398,0.500,0.220,42,176,None,None
1,y_stress_1d,val,0.639,0.445,0.578,0.252,28,82,None,None
2,y_stress_1d,test,0.554,0.476,0.462,0.254,29,72,None,"[0.351, 0.738]"


## Regressors — log1p(next-day liquidation debt covered USD), tuned

In [22]:
res_ridge_reg = mtr.run_regressor("ridge", data)
mps.save_model_result(res_ridge_reg)
mt.log_model_result(res_ridge_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_ridge_reg))

 ridge: tuned over 3 configs -> {'alpha': 100.0} (CV RMSE 4.320)
 ridge: train/val/test R² 0.233/0.041/-0.051 | test MAE(log) 3.890 | MAE $6,108,780
 saved ridge__regressor → model_results/
    mlflow: logged ridge__regressor (13 metrics) run 5957fe57
🏃 View run ridge__regressor at: http://localhost:5001/#/experiments/1/runs/5957fe57a170462e80af79eb64f15043
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,3.974,3.323,0.233,827762.0
1,val,5.183,4.540,0.041,3660138.0
2,test,4.641,3.890,-0.051,6108780.0


In [23]:
res_lasso_reg = mtr.run_regressor("lasso", data)
mps.save_model_result(res_lasso_reg)
mt.log_model_result(res_lasso_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_lasso_reg))

 lasso: tuned over 3 configs -> {'alpha': 1.0} (CV RMSE 4.225)
 lasso: train/val/test R² 0.105/0.055/-0.095 | test MAE(log) 3.971 | MAE $6,160,705
 saved lasso__regressor → model_results/
    mlflow: logged lasso__regressor (13 metrics) run 01914b9f
🏃 View run lasso__regressor at: http://localhost:5001/#/experiments/1/runs/01914b9fce9b4e1784c0200f4f106e68
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,4.294,3.599,0.105,833054.0
1,val,5.145,4.435,0.055,3141611.0
2,test,4.736,3.971,-0.095,6160705.0


In [24]:
res_kneighbors_reg = mtr.run_regressor("kneighbors", data)
mps.save_model_result(res_kneighbors_reg)
mt.log_model_result(res_kneighbors_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_kneighbors_reg))

 kneighbors: tuned over 3 configs -> {'n_neighbors': 15} (CV RMSE 4.125)
 kneighbors: train/val/test R² 0.166/0.001/-0.190 | test MAE(log) 4.132 | MAE $6,159,509
 saved kneighbors__regressor → model_results/
    mlflow: logged kneighbors__regressor (13 metrics) run 87d4810c
🏃 View run kneighbors__regressor at: http://localhost:5001/#/experiments/1/runs/87d4810caa0248f19e1b3203c32c1d2e
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,4.145,3.451,0.166,831859.0
1,val,5.290,4.620,0.001,3140284.0
2,test,4.937,4.132,-0.190,6159509.0


In [25]:
res_svr_rbf_reg = mtr.run_regressor("svr_rbf", data)
mps.save_model_result(res_svr_rbf_reg)
mt.log_model_result(res_svr_rbf_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_svr_rbf_reg))

 svr_rbf: tuned over 4 configs -> {'C': 1.0, 'epsilon': 0.5} (CV RMSE 4.231)
 svr_rbf: train/val/test R² 0.267/0.048/-0.136 | test MAE(log) 4.001 | MAE $6,162,168
 saved svr_rbf__regressor → model_results/
    mlflow: logged svr_rbf__regressor (13 metrics) run 31eaa554
🏃 View run svr_rbf__regressor at: http://localhost:5001/#/experiments/1/runs/31eaa554fc8146f3bb1e95fe5325d4d0
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,3.885,3.116,0.267,826567.0
1,val,5.165,4.425,0.048,3140509.0
2,test,4.825,4.001,-0.136,6162168.0


In [26]:
res_random_forest_reg = mtr.run_regressor("random_forest", data)
mps.save_model_result(res_random_forest_reg)
mt.log_model_result(res_random_forest_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_random_forest_reg))

 random_forest: tuned over 4 configs -> {'max_depth': None, 'min_samples_leaf': 5} (CV RMSE 4.400)
 random_forest: train/val/test R² 0.676/0.065/-0.354 | test MAE(log) 4.407 | MAE $6,162,617
 saved random_forest__regressor → model_results/
    mlflow: logged random_forest__regressor (13 metrics) run 9ea3f181
🏃 View run random_forest__regressor at: http://localhost:5001/#/experiments/1/runs/9ea3f18194c3488c90c374ce11291505
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,2.584,2.090,0.676,806394.0
1,val,5.117,4.473,0.065,3138086.0
2,test,5.266,4.407,-0.354,6162617.0


In [27]:
res_extra_trees_reg = mtr.run_regressor("extra_trees", data)
mps.save_model_result(res_extra_trees_reg)
mt.log_model_result(res_extra_trees_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_extra_trees_reg))

 extra_trees: tuned over 4 configs -> {'max_depth': 6, 'min_samples_leaf': 5} (CV RMSE 4.392)
 extra_trees: train/val/test R² 0.608/0.054/-0.244 | test MAE(log) 4.197 | MAE $6,161,846
 saved extra_trees__regressor → model_results/
    mlflow: logged extra_trees__regressor (13 metrics) run 94db9f48
🏃 View run extra_trees__regressor at: http://localhost:5001/#/experiments/1/runs/94db9f48d65e43fd8718dca645fbb380
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,2.840,2.332,0.608,806242.0
1,val,5.147,4.496,0.054,3135073.0
2,test,5.048,4.197,-0.244,6161846.0


In [28]:
res_hist_gradient_boosting_reg = mtr.run_regressor("hist_gradient_boosting", data)
mps.save_model_result(res_hist_gradient_boosting_reg)
mt.log_model_result(res_hist_gradient_boosting_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_hist_gradient_boosting_reg))

 hist_gradient_boosting: tuned over 4 configs -> {'learning_rate': 0.05, 'max_leaf_nodes': 7} (CV RMSE 4.269)
 hist_gradient_boosting: train/val/test R² 0.799/0.021/-0.288 | test MAE(log) 4.245 | MAE $6,166,406
 saved hist_gradient_boosting__regressor → model_results/
    mlflow: logged hist_gradient_boosting__regressor (13 metrics) run 70f23bda
🏃 View run hist_gradient_boosting__regressor at: http://localhost:5001/#/experiments/1/runs/70f23bda99624d07b498b3132eb15ac7
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,2.034,1.569,0.799,748127.0
1,val,5.237,4.420,0.021,3131220.0
2,test,5.136,4.245,-0.288,6166406.0


In [29]:
res_xgboost_reg = mtr.run_regressor("xgboost", data)
mps.save_model_result(res_xgboost_reg)
mt.log_model_result(res_xgboost_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_xgboost_reg))

 xgboost: tuned over 4 configs -> {'max_depth': 4, 'learning_rate': 0.03} (CV RMSE 4.610)
 xgboost: train/val/test R² 0.995/-0.009/-0.308 | test MAE(log) 4.355 | MAE $6,163,475
 saved xgboost__regressor → model_results/
    mlflow: logged xgboost__regressor (13 metrics) run 56bf3652
🏃 View run xgboost__regressor at: http://localhost:5001/#/experiments/1/runs/56bf3652e96949e48e07a7edf366bd15
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,0.329,0.250,0.995,250738.0
1,val,5.318,4.483,-0.009,3136454.0
2,test,5.177,4.355,-0.308,6163475.0


In [30]:
res_lightgbm_reg = mtr.run_regressor("lightgbm", data)
mps.save_model_result(res_lightgbm_reg)
mt.log_model_result(res_lightgbm_reg, SPLIT_META, RUN_TAGS)
display(mbench.model_scorecard(res_lightgbm_reg))

 lightgbm: tuned over 4 configs -> {'max_depth': -1, 'learning_rate': 0.03} (CV RMSE 4.403)
 lightgbm: train/val/test R² 0.951/-0.008/-0.307 | test MAE(log) 4.276 | MAE $6,171,945
 saved lightgbm__regressor → model_results/
    mlflow: logged lightgbm__regressor (13 metrics) run 3a01b4a8
🏃 View run lightgbm__regressor at: http://localhost:5001/#/experiments/1/runs/3a01b4a807f4409db244ab0a9c48e106
🧪 View experiment at: http://localhost:5001/#/experiments/1


,split,rmse_log,mae_log,r2,mae_usd
0,train,1.002,0.767,0.951,625822.0
1,val,5.316,4.437,-0.008,3116256.0
2,test,5.174,4.276,-0.307,6171945.0


## Meta-learner — Protocol Risk Index (PRI)

In [31]:
from pathlib import Path

# blend every trained component (skill-shrunk weights; baselines excluded)
results = mps.load_model_results()
res_pri = mtr.run_meta_pri(results, data)
mps.save_model_result(res_pri)
# logs the blend weights too — log_model_result turns res_pri["weights"] into
# one weight_<component> metric each, so blend composition is comparable across runs
mt.log_model_result(res_pri, SPLIT_META, RUN_TAGS)
res_pri["pri_frame"].to_csv(Path(cfg.MODEL_RESULTS_DIR) / "pri_timeseries.csv", index=False)
print(" wrote pri_timeseries.csv")

conc = mbench.weight_concentration(res_pri["weights"])
print(f" blend concentration: {conc['n_nonzero']} non-zero of {res_pri['n_components']} "
      f"components | effective_n {conc['effective_n']} | max weight {conc['max_weight']}")
display(mbench.model_scorecard(res_pri))

 meta_pri: 15 components | test ROC-AUC 0.540 | PR-AUC 0.457
 weights: {'transformer': 0.106, 'lstm': 0.098, 'xgboost': 0.092, 'extra_trees': 0.092, 'random_forest': 0.086, 'cox_ph': 0.076, 'mlp': 0.073, 'hist_gradient_boosting': 0.07, 'lightgbm': 0.069, 'kneighbors': 0.053, 'gaussian_nb': 0.047, 'svc_rbf': 0.041, 'isolation_forest': 0.032, 'logistic_regression': 0.032, 'ridge_classifier': 0.032}
 saved meta_pri__meta → model_results/
    mlflow: logged meta_pri__meta (47 metrics) run 0bac47f8
🏃 View run meta_pri__meta at: http://localhost:5001/#/experiments/1/runs/0bac47f8c8144a02810821659b2273e5
🧪 View experiment at: http://localhost:5001/#/experiments/1
 wrote pri_timeseries.csv
 blend concentration: 15 non-zero of 15 components | effective_n 13.2 | max weight 0.106


,target,split,roc_auc,pr_auc,f1,brier,n_pos,n,cv_auc,auc_ci95
0,y_stress_1d,train,0.923,0.795,0.662,None,47,189,None,None
1,y_stress_1d,val,0.639,0.552,0.494,None,28,82,None,None
2,y_stress_1d,test,0.540,0.457,0.507,None,29,72,None,"[0.389, 0.694]"
3,y_stress_3d,train,0.809,0.526,0.590,None,47,189,None,None
4,y_stress_3d,val,0.586,0.547,0.506,None,30,82,None,None
5,y_stress_3d,test,0.575,0.472,0.493,None,27,72,None,None
6,y_stress_7d,train,0.719,0.443,0.493,None,46,189,None,None
7,y_stress_7d,val,0.659,0.587,0.568,None,31,82,None,None
8,y_stress_7d,test,0.539,0.762,0.687,None,53,72,None,None


In [32]:
from pathlib import Path
files = sorted(Path(cfg.MODEL_RESULTS_DIR).iterdir())
all_res = mps.load_model_results()
trained = [r["name"] for r in all_res.values() if r.get("status") == "trained"]
skipped = [r["name"] for r in all_res.values() if r.get("status") != "trained"]
print(f" {len(files)} files in {cfg.MODEL_RESULTS_DIR}/")
print(f" trained: {len(trained)} -> {trained}")
print(f" skipped/failed: {skipped or 'none'}")

 60 files in model_results/
 trained: 29 -> ['baseline_median', 'baseline_persistence', 'baseline_persistence', 'baseline_prevalence', 'cox_ph', 'extra_trees', 'extra_trees', 'gaussian_nb', 'hist_gradient_boosting', 'hist_gradient_boosting', 'isolation_forest', 'kneighbors', 'kneighbors', 'lasso', 'lightgbm', 'lightgbm', 'logistic_regression', 'lstm', 'meta_pri', 'mlp', 'random_forest', 'random_forest', 'ridge', 'ridge_classifier', 'svc_rbf', 'svr_rbf', 'transformer', 'xgboost', 'xgboost']
 skipped/failed: none
